# AutoFuse简介

本节围绕"什么是 AutoFuse""接入路线""融合过程"三条主线展开，并补充产品支持型号与使用边界。为了便于初学者理解，本节先说明通用术语与相关概念，再补充课程化解释，帮助你建立 AutoFuse 的整体认知。

本节学习大纲如下：

- 通用术语与相关概念
- 什么是 AutoFuse
- AutoFuse 接入路线
- AutoFuse 融合过程
- 产品支持型号与使用边界


## 1. 通用术语与相关概念
为便于理解后续内容，学习本课程前请先了解如下术语、缩略语及相关概念。

<table align="left">
  <thead>
    <tr>
      <th>术语</th>
      <th>说明</th>
      <th>在本课程中的理解方式</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>算子</td>
      <td>Operator，深度学习模型中执行具体计算的基本单元</td>
      <td>AutoFuse 识别和融合的基本对象</td>
    </tr>
    <tr>
      <td>融合算子</td>
      <td>Fused Operator，将多个相邻算子合并后形成的算子</td>
      <td>AutoFuse 融合后的计算形式</td>
    </tr>
    <tr>
      <td>NPU</td>
      <td>Neural Processing Unit，神经网络处理器</td>
      <td>昇腾 AI 处理器执行计算的硬件基础</td>
    </tr>
    <tr>
      <td>Host</td>
      <td>主机侧</td>
      <td>负责模型编译、准备参数和训练/推理进程管理</td>
    </tr>
    <tr>
      <td>Device</td>
      <td>昇腾设备侧</td>
      <td>负责模型内算子的调度和执行</td>
    </tr>
    <tr>
      <td>GM</td>
      <td>Global Memory，全局内存</td>
      <td>设备侧容量较大但访问代价更高的存储空间，作为算子间数据流转存储单元</td>
    </tr>
    <tr>
      <td>UB</td>
      <td>Unified Buffer，统一缓冲区</td>
      <td>AI Core 片上缓存空间，容量小但访问速度比 GM 快几十倍，作为算子内数据流转存储单元</td>
    </tr>
    <tr>
      <td>MTE</td>
      <td>Memory Transfer Engine，AI Core 中与数据搬运相关的能力</td>
      <td>搬运数据的硬件通道，用于在 GM 与 UB 间完成数据搬运</td>
    </tr>
    <tr>
      <td>Vector 计算</td>
      <td>对向量数据同时做相同运算的计算方式</td>
      <td>AutoFuse 常见收益场景之一，多个小 Vector 计算可能带来较多搬运开销</td>
    </tr>
    <tr>
      <td>Memory Bound</td>
      <td>性能主要受数据搬运限制，而不是受计算能力限制</td>
      <td>数据搬运耗时是性能瓶颈，而不是计算耗时</td>
    </tr>
    <tr>
      <td>图编译</td>
      <td>将模型图转换为昇腾可执行模型或编译产物的过程</td>
      <td>AutoFuse 主要发挥作用的阶段</td>
    </tr>
    <tr>
      <td>OM 模型</td>
      <td>Offline Model，昇腾离线模型文件</td>
      <td>通过 ATC 触发离线编译后常见的模型交付件</td>
    </tr>
    <tr>
      <td>GE</td>
      <td>Graph Engine，昇腾图引擎</td>
      <td>模型编译加速组件，使能 AutoFuse 的场景之一</td>
    </tr>
    <tr>
      <td>ATC</td>
      <td>Ascend Tensor Compiler，昇腾模型转换/离线编译工具</td>
      <td>常见的离线编译入口，底层会进入 GE 图编译相关流程</td>
    </tr>
    <tr>
      <td>IR</td>
      <td>Intermediate Representation，中间表示</td>
      <td>编译器内部用于描述计算逻辑的表达形式</td>
    </tr>
    <tr>
      <td>AscIR</td>
      <td>Ascend C IR，面向 Ascend C 语言建模的中间表示</td>
      <td>衔接前端图表示与后端算子代码的中间桥梁</td>
    </tr>
    <tr>
      <td>AutoFuse</td>
      <td>基于 Ascend C 的自动融合框架</td>
      <td>在图编译阶段识别可融合范围，并生成融合算子相关实现</td>
    </tr>
    <tr>
      <td>融合策略</td>
      <td>判断哪些结构适合融合的规则和求解过程</td>
      <td>避免把所有相邻算子都盲目合并</td>
    </tr>
    <tr>
      <td>FusedGraph</td>
      <td>表示一个融合范围的图结构</td>
      <td>AutoFuse 前端识别出的可融合算子集合</td>
    </tr>
    <tr>
      <td>AscBackend</td>
      <td>AutoFuse 融合后形成的 Ascend IR 算子节点</td>
      <td>承载融合子图结构，携带 AscGraph 属性信息</td>
    </tr>
    <tr>
      <td>AscGraph</td>
      <td>AscBackend 节点携带的子图对象</td>
      <td>内部包含多个 AscIR 节点，用于描述融合后的计算结构</td>
    </tr>
    <tr>
      <td>Schedule</td>
      <td>算子调度优化</td>
      <td>对融合后的计算逻辑进行组织优化，调整执行顺序、合并循环、复用缓存，在不改变结果的前提下提升性能</td>
    </tr>
    <tr>
      <td>Tiling</td>
      <td>将大 shape 的计算任务切分为适配硬件缓存大小的小块</td>
      <td>融合性能优化的关键技术</td>
    </tr>
    <tr>
      <td>Auto Tiling</td>
      <td>自动切分</td>
      <td>评估多种切分方案的性能，自动选出使 Kernel 执行性能最优的 Tiling 策略</td>
    </tr>
    <tr>
      <td>Lowering</td>
      <td>表达层级转换</td>
      <td>将较高层的图语义转换为后端更容易处理的 AscIR 表达</td>
    </tr>
    <tr>
      <td>Codegen</td>
      <td>Code Generation，代码生成</td>
      <td>解析调度结果，生成 Host 侧 Tiling 代码和 Device 侧 Kernel 代码</td>
    </tr>
    <tr>
      <td>Ascend C</td>
      <td>面向昇腾 AI 处理器的算子开发语言</td>
      <td>AutoFuse 后端生成融合算子代码时依赖的目标语言</td>
    </tr>
    <tr>
      <td>BiSheng Compiler</td>
      <td>毕昇编译器，昇腾 AI 软件栈中的编译器组件</td>
      <td>AutoFuse 生成的融合 kernel 源码通过毕昇编译器编译为 NPU 可执行的二进制产物</td>
    </tr>
    <tr>
      <td>Kernel</td>
      <td>在 NPU 上实际执行计算的代码片段</td>
      <td>AutoFuse 后端生成的融合计算单元</td>
    </tr>
    <tr>
      <td>Dynamic Shape</td>
      <td>动态 shape</td>
      <td>模型输入或中间张量的某些维度在编译时不能完全固定</td>
    </tr>
    <tr>
      <td>Dynamo</td>
      <td>PyTorch Dynamo，PyTorch 生态中的动态图捕获与图生成组件</td>
      <td>可理解为 PyTorch 编译链路的前端入口，负责捕获 Python 模型执行并生成可交给后续编译器处理的图表示</td>
    </tr>
    <tr>
      <td>PyTorch Inductor</td>
      <td>PyTorch 生态中的编译优化组件</td>
      <td>使能 AutoFuse 的另一条重要对接路线</td>
    </tr>
  </tbody>
</table>
<div style="clear:left"></div>


## 2. 什么是 AutoFuse

AutoFuse 是基于 Ascend C 的自动融合框架，支持自动融合范围识别、自动算子代码生成、Auto Tiling 优化及 Dynamic Shape 等特性。

在算法网络中，由于存在大量 Vector 计算，各个 Vector 计算之间会产生大量内存搬运，导致 Memory Bound 问题。AutoFuse 通过自动将多个算子融合为一个算子，减少网络中的算子数量和内存搬运，从而缓解 Memory Bound 问题，释放昇腾算力，提升模型执行性能。

<div style="text-align:left">
<img src="./images/autofuse_benefit_principle.png" alt="AutoFuse 收益原理" width="40%">
</div>

结合课程视角，可以这样理解：AutoFuse 并不是让计算本身消失，而是将多个满足条件的小算子组织成一个融合算子，让中间结果尽量在片上缓存中流转，减少重复搬运和多次调度。从收益机制看，自动融合理论上在 MTE 搬运、Dynamic Shape 调度开销方面都会有一定收益；对于小 shape、MTE Bound 的推荐网络，一般更容易获得正收益。

## 3. AutoFuse 接入路线

AutoFuse 自动融合方案基于昇腾 NPU 底层统一的 Ascend C IR 与代码生成能力，目前提供了两条接入路径。

<div style="text-align:left">
<img src="./images/autofuse_technical_route.png" alt="AutoFuse 接入路线" width="40%">
</div>
第一条路径基于昇腾自研 GE 框架，注重 NPU 亲和性。该路径包含三部分核心能力：

- Ascend IR 的符号化 shape 推导，通过变量符号表达动态变化的 shape，从而在编译时基于符号化的 shape 进行代码生成。
- Ascend IR 到 AscIR 的 Lowering 实现，使用低层级的 AscIR 表达 Ascend IR 的计算逻辑，确定融合结构。
- 融合策略，结合 AscIR 的特点与约束，进行融合结构间的循环轴合并，获得融合最优解。

第二条路径对接 PyTorch Inductor，聚焦生态支持，复用 Inductor 的融合能力，并将 Inductor IR 表达的融合结构转换为 Ascend C IR 图进行代码生成。算子融合方式与第一条路径类似，当前路径功能已支持，但是还未正式商用。

## 4. AutoFuse 融合过程

AutoFuse 自动融合的实现过程可以划分为两部分：

- 自动确定融合范围。
- 根据融合范围自动生成融合 Kernel 执行源码及 Tiling 计算源码。

前者称为自动融合前端，后者称为自动融合后端（对应上图中的公共底层能力）。前端主要根据规则或配置判断哪些算子能够融合，并确定一个融合算子的融合范围；融合范围使用 FusedGraph 表达。

<div style="text-align:left">
<img src="./images/fusedgraph_structure.png" alt="FusedGraph" width="28%">
</div>

FusedGraph 内部包含一个或多个 AscBackend 节点。一个 AscBackend 节点携带一个 AscGraph 属性，一个 AscGraph 内包含多个 AscIR 节点。

<div style="text-align:left">
<img src="./images/ascgraph_structure.png" alt="AscBackend 对应的 AscGraph" width="45%">
</div>

后端接收到 FusedGraph 后，根据融合范围自动生成融合 Kernel 执行源码及 Tiling 计算源码。这里先帮助读者建立整体流程认知，自动融合原理会在后续章节体现。

## 5. 产品支持型号与使用边界

AutoFuse 自动融合特性当前支持以下产品型号：

- Atlas 350 加速卡
- Atlas A3 训练系列产品 / Atlas A3 推理系列产品
- Atlas A2 训练系列产品 / Atlas A2 推理系列产品

除了产品型号，还需要关注 AutoFuse 的使用边界：

- **开启方式**：通过 `AUTOFUSE_FLAGS` 在图编译阶段开启；基础配置方式会在下一节说明。
- **触发方式**：AutoFuse 依赖图编译流程触发，使用者无需在业务代码中直接调用其接口。
  - 基于 GE 框架的图编译路径，由 GE/ATC 编译流程在满足条件时自动调用 AutoFuse 融合流程。
  - 对接 PyTorch Inductor 的编译路径（通过 Dynamo 捕获模型图并交由 Inductor 编译），由 PyTorch 编译入口触发 AutoFuse 融合流程。
- **收益不确定性**：是否获得性能收益需要结合模型结构、shape、算子类型和实际性能分析判断，无法保证所有模型开启后都有提升。

## 6. 课后练习

本节介绍了 AutoFuse 的基本定义、接入路线、融合过程和使用边界，请根据学习内容完成以下题目进行自测。

1. （判断题）AutoFuse 主要在图编译阶段发挥作用，通过自动识别融合范围并生成融合算子相关实现，减少算子数量和内存搬运。

2. （判断题）只要开启 AutoFuse，所有模型都一定能获得性能提升。

3. （单选题）AutoFuse 缓解 Memory Bound 问题的主要方式是什么？
    A. 将所有计算转移到 CPU 上执行
    B. 将多个满足条件的算子融合为一个算子，减少中间数据搬运和调度开销
    C. 删除模型中的所有小算子
    D. 只改变模型文件名，不改变编译过程

4. （单选题）如何理解 AutoFuse 的两条接入路线？
    A. 两条路线都体现 AutoFuse 自动融合的关键思想，课程从原理出发理解整体链路
    B. 只有基于 GE 框架的路径属于自动融合，另一条路径只是生态适配，不涉及融合
    C. PyTorch Inductor 路径已经作为商用交付能力，使用者必须优先选择该路径
    D. 必须由业务代码直接调用 AutoFuse 内部后端接口，否则无法触发自动融合

5. （多选题）从普通使用者视角看，学习 AutoFuse 时需要重点关注哪些内容？
    A. 如何通过 `AUTOFUSE_FLAGS` 开启相关能力
    B. AutoFuse 依赖 GE 图编译流程生效，通过 ATC 触发离线编译生成 OM 是常见场景，在线 GE 编译路径满足条件时也可能生效
    C. 需要结合编译日志、融合产物、性能数据和精度对比判断融合效果
    D. 必须由业务代码直接调用 AutoFuse 内部后端接口，否则编译链路无法触发自动融合

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/01.02_answer.txt